# Week 69, Day 1: Project: an interpreter

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w069d1.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


> **Key idea.** An interpreter runs programs in stages: a tokeniser turns characters into tokens, a parser turns tokens into a syntax tree, and an evaluator walks the tree. Every stage reports errors with a position, through one error family.

### Exercise: Step 1: the error family

Write `TinyError(message, line=None, column=None)`, an `Exception` subclass that stores `message`, `line` and `column`, and whose `str()` is `"line 3, column 7: unexpected character '@'"`, or `"line 3: division by zero"` when there's no column, or just the message when there's no line. Then three subclasses with docstrings: `LexError`, `ParseError` and `TinyRuntimeError` (not `RuntimeError`, which is Python's own).

In [ ]:
class TinyError(Exception):
    ___

class LexError(TinyError):
    ___

class ParseError(TinyError):
    ___

class TinyRuntimeError(TinyError):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_errors():\n    \"the error family\"\n    e = LexError(\"unexpected character '@'\", 3, 7)\n    assert str(e) == \"line 3, column 7: unexpected character '@'\" and e.line == 3 and e.column == 7\n    assert e.message == \"unexpected character '@'\" and isinstance(e, TinyError) and isinstance(e, Exception)\n    assert str(TinyRuntimeError(\"division by zero\", 2)) == \"line 2: division by zero\"\n    assert str(ParseError(\"oops\")) == \"oops\" and ParseError(\"oops\").line is None\n    assert issubclass(TinyRuntimeError, TinyError) and not issubclass(TinyRuntimeError, RuntimeError)\n    for cls in (LexError, ParseError, TinyRuntimeError):\n        assert cls.__doc__\n    try:\n        raise ParseError(\"expected ';'\", 1, 5)\n    except TinyError as caught:\n        assert caught.column == 5")

### Exercise: Step 2: showing an error

Good error messages show where the problem is. Write `show_error(source, error)`: for an error with a line and column, return three lines: the error's `str()`, then the offending source line indented by four spaces, then a caret `^` under the column (four spaces plus `column - 1` spaces). Without a column, return just the first two lines; without a line, just `str(error)`. The error classes from Step 1 are given.

In [ ]:
class TinyError(Exception):
    """Base class for every error the Tiny interpreter reports."""

    def __init__(self, message, line=None, column=None):
        super().__init__(message)
        self.message, self.line, self.column = message, line, column

    def __str__(self):
        if self.line is None:
            return self.message
        if self.column is None:
            return f"line {self.line}: {self.message}"
        return f"line {self.line}, column {self.column}: {self.message}"


class LexError(TinyError):
    """The source contains characters that don't form a token."""


class ParseError(TinyError):
    """The tokens don't follow Tiny's grammar."""


class TinyRuntimeError(TinyError):
    """Something went wrong while a program ran."""


def show_error(source, error):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_show():\n    \"error display\"\n    src = \"let x = 1;\\nlet y = 3 @ 4;\\n\"\n    assert show_error(src, LexError(\"unexpected character '@'\", 2, 11)) == (\n        \"line 2, column 11: unexpected character '@'\\n    let y = 3 @ 4;\\n              ^\")\n    assert show_error(src, TinyRuntimeError(\"division by zero\", 1)) == \"line 1: division by zero\\n    let x = 1;\"\n    assert show_error(src, ParseError(\"no line\")) == \"no line\"\n    assert show_error(\"x\", LexError(\"bad\", 1, 1)).endswith(\"\\n    ^\")")

### Exercise: Step 3: tokens and a cursor

Write `Token`, a **frozen** dataclass with `kind` (such as `"NUMBER"`), `text` (exactly as written), `value` (its meaning: the number, the string's contents...), `line` and `column`. Then `Cursor(source)`, which walks through the text keeping its position: `pos` (starting at 0), `line` and `column` (both starting at 1); `at_end()`; `peek(ahead=0)`, the character `ahead` places from the current one, or `""` past the end; and `advance()`, which returns the current character and moves on, going to column 1 of the next line after a newline.

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Token:
    ___

class Cursor:
    def __init__(self, source):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_tokens():\n    \"tokens and positions\"\n    import dataclasses\n    t = Token(\"NUMBER\", \"42\", 42, 1, 5)\n    assert (t.kind, t.text, t.value, t.line, t.column) == (\"NUMBER\", \"42\", 42, 1, 5)\n    try:\n        t.value = 7\n        assert False\n    except dataclasses.FrozenInstanceError:\n        pass\n    c = Cursor(\"ab\\ncd\")\n    assert (c.pos, c.line, c.column) == (0, 1, 1) and c.peek() == \"a\" and c.peek(1) == \"b\" and c.peek(9) == \"\"\n    assert c.advance() == \"a\" and c.advance() == \"b\" and (c.line, c.column) == (1, 3)\n    assert c.advance() == \"\\n\" and (c.line, c.column) == (2, 1) and c.peek() == \"c\"\n    c.advance(); c.advance()\n    assert c.at_end() and c.peek() == \"\" and (c.line, c.column, c.pos) == (2, 3, 5)\n    assert Cursor(\"\").at_end()")

### Exercise: Step 4: numbers

Write `read_number(cursor)`: with the cursor at a digit, read the whole number and return a `NUMBER` token positioned where it **started**: whole numbers like `42` get an `int` value, and numbers with a decimal part like `3.25` a `float`. A dot only belongs to the number if a digit follows it, so `3.` reads as `3` (the dot is left for later) and `1.5.2` reads as `1.5`. `Token` and `Cursor` are given.

In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class Token:
    kind: str
    text: str
    value: object
    line: int
    column: int


class Cursor:
    def __init__(self, source):
        self.source, self.pos, self.line, self.column = source, 0, 1, 1

    def at_end(self):
        return self.pos >= len(self.source)

    def peek(self, ahead=0):
        i = self.pos + ahead
        return self.source[i] if i < len(self.source) else ""

    def advance(self):
        ch = self.source[self.pos]
        self.pos += 1
        if ch == "\n":
            self.line, self.column = self.line + 1, 1
        else:
            self.column += 1
        return ch


def read_number(cursor):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_numbers():\n    \"reading numbers\"\n    c = Cursor(\"42 + 7\")\n    assert read_number(c) == Token(\"NUMBER\", \"42\", 42, 1, 1) and c.peek() == \" \"\n    c = Cursor(\"x = 3.25;\")\n    c.advance(); c.advance(); c.advance(); c.advance()\n    t = read_number(c)\n    assert t == Token(\"NUMBER\", \"3.25\", 3.25, 1, 5) and type(t.value) is float and c.peek() == \";\"\n    c = Cursor(\"3.\")\n    assert read_number(c).value == 3 and c.peek() == \".\"\n    c = Cursor(\"1.5.2\")\n    assert read_number(c).text == \"1.5\" and c.peek() == \".\"\n    c = Cursor(\"\\n  007\")\n    c.advance(); c.advance(); c.advance()\n    assert read_number(c) == Token(\"NUMBER\", \"007\", 7, 2, 3)")

### Exercise: Step 5: strings

Write `read_string(cursor)`: with the cursor at a `"`, read a string up to its closing quote and return a `STRING` token whose `text` is exactly as written (quotes and backslashes included) and whose `value` is its contents, with the escapes in `ESCAPES` turned into the characters they stand for (`\\n` into a newline, and so on). Errors, as `LexError`: a string that reaches a newline or the end of the source is `"unterminated string"`, positioned at the **opening** quote; an unknown escape such as `\\q` is `"unknown escape \\q"` (with one backslash in the message), positioned at its backslash. The earlier pieces are given.

In [ ]:
from dataclasses import dataclass


class TinyError(Exception):
    """Base class for every error the Tiny interpreter reports."""

    def __init__(self, message, line=None, column=None):
        super().__init__(message)
        self.message, self.line, self.column = message, line, column

    def __str__(self):
        if self.line is None:
            return self.message
        if self.column is None:
            return f"line {self.line}: {self.message}"
        return f"line {self.line}, column {self.column}: {self.message}"


class LexError(TinyError):
    """The source contains characters that don't form a token."""


class ParseError(TinyError):
    """The tokens don't follow Tiny's grammar."""


class TinyRuntimeError(TinyError):
    """Something went wrong while a program ran."""


@dataclass(frozen=True)
class Token:
    kind: str
    text: str
    value: object
    line: int
    column: int


class Cursor:
    def __init__(self, source):
        self.source, self.pos, self.line, self.column = source, 0, 1, 1

    def at_end(self):
        return self.pos >= len(self.source)

    def peek(self, ahead=0):
        i = self.pos + ahead
        return self.source[i] if i < len(self.source) else ""

    def advance(self):
        ch = self.source[self.pos]
        self.pos += 1
        if ch == "\n":
            self.line, self.column = self.line + 1, 1
        else:
            self.column += 1
        return ch


ESCAPES = {"n": "\n", "t": "\t", '"': '"', "\\": "\\"}


def read_string(cursor):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_strings():\n    \"reading strings\"\n    c = Cursor('\"hello\" rest')\n    assert read_string(c) == Token(\"STRING\", '\"hello\"', \"hello\", 1, 1) and c.peek() == \" \"\n    c = Cursor(r'\"a\\tb\\n\\\"q\\\" \\\\\"')\n    t = read_string(c)\n    assert t.value == 'a\\tb\\n\"q\" \\\\' and t.text == r'\"a\\tb\\n\\\"q\\\" \\\\\"' and c.at_end()\n    assert read_string(Cursor('\"\"')).value == \"\"\n    for src, message, column in (('\"oops', \"unterminated string\", 1), ('\"line\\nbreak\"', \"unterminated string\", 1),\n                                 (r'\"bad \\q\"', r\"unknown escape \\q\", 6)):\n        try:\n            read_string(Cursor(src))\n            assert False, src\n        except LexError as e:\n            assert e.message == message and e.column == column and e.line == 1")

In [ ]:
from dataclasses import dataclass


class TinyError(Exception):
    """Base class for every error the Tiny interpreter reports."""

    def __init__(self, message, line=None, column=None):
        super().__init__(message)
        self.message, self.line, self.column = message, line, column

    def __str__(self):
        if self.line is None:
            return self.message
        if self.column is None:
            return f"line {self.line}: {self.message}"
        return f"line {self.line}, column {self.column}: {self.message}"


class LexError(TinyError):
    """The source contains characters that don't form a token."""


class ParseError(TinyError):
    """The tokens don't follow Tiny's grammar."""


class TinyRuntimeError(TinyError):
    """Something went wrong while a program ran."""


def show_error(source, error):
    if error.line is None:
        return str(error)
    text = source.split("\n")[error.line - 1]
    if error.column is None:
        return f"{error}\n    {text}"
    return f"{error}\n    {text}\n    {' ' * (error.column - 1)}^"


@dataclass(frozen=True)
class Token:
    kind: str
    text: str
    value: object
    line: int
    column: int


class Cursor:
    def __init__(self, source):
        self.source, self.pos, self.line, self.column = source, 0, 1, 1

    def at_end(self):
        return self.pos >= len(self.source)

    def peek(self, ahead=0):
        i = self.pos + ahead
        return self.source[i] if i < len(self.source) else ""

    def advance(self):
        ch = self.source[self.pos]
        self.pos += 1
        if ch == "\n":
            self.line, self.column = self.line + 1, 1
        else:
            self.column += 1
        return ch


def read_number(cursor):
    line, column, text = cursor.line, cursor.column, ""
    while cursor.peek().isdigit():
        text += cursor.advance()
    if cursor.peek() == "." and cursor.peek(1).isdigit():
        text += cursor.advance()
        while cursor.peek().isdigit():
            text += cursor.advance()
        return Token("NUMBER", text, float(text), line, column)
    return Token("NUMBER", text, int(text), line, column)


ESCAPES = {"n": "\n", "t": "\t", '"': '"', "\\": "\\"}


def read_string(cursor):
    line, column = cursor.line, cursor.column
    text = cursor.advance()
    value = ""
    while True:
        if cursor.at_end() or cursor.peek() == "\n":
            raise LexError("unterminated string", line, column)
        ch = cursor.advance()
        text += ch
        if ch == '"':
            return Token("STRING", text, value, line, column)
        if ch == "\\":
            if cursor.at_end():
                raise LexError("unterminated string", line, column)
            esc_line, esc_col = cursor.line, cursor.column - 1
            code = cursor.advance()
            text += code
            if code not in ESCAPES:
                raise LexError(f"unknown escape \\{code}", esc_line, esc_col)
            value += ESCAPES[code]
        else:
            value += ch


line = 'print 3.5; print "tab\there"; print "oops'
cursor = Cursor(line)
found = []
try:
    while not cursor.at_end():
        ch = cursor.peek()
        if ch.isdigit():
            found.append(read_number(cursor))
        elif ch == '"':
            found.append(read_string(cursor))
        else:
            cursor.advance()
except TinyError as e:
    print(show_error(line, e))
for tok in found:
    print(tok.kind, repr(tok.value), "at column", tok.column)

> **Why errors first?.** It's tempting to start with the interesting part, running programs, and add error handling later. But in an interpreter, errors are half the user interface: most of the time someone runs a program, it's because they're still writing it and it doesn't work yet. Deciding today that every error has a class, a message and a position means each later stage has an obvious way to report problems, and the person using Tiny always gets told where to look.

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

## The brief

You'll write an interpreter for **Tiny**, a small programming language. This is a complete Tiny program:

```text
# Tiny: comments start with #
fn make_counter() {
    let count = 0;
    fn step() {
        count = count + 1;
        return count;
    }
    return step;
}

let next = make_counter();
next();
next();
print next();                      # 3

fn fib(n) {
    if (n < 2) return n;
    return fib(n - 1) + fib(n - 2);
}
print "fib(15) is " + str(fib(15));

let i = 0;
while (i < 3) {
    print i * i;
    i = i + 1;
}
```

The language, in full:

- **Values**: numbers (`42`, `3.5`), strings (`"text"`, with the escapes `\n`, `\t`, `\"` and `\\`), `true`, `false`, `nil`, and functions.
- **Expressions**: arithmetic `+ - * / %` (with `+` also joining strings), comparisons `< <= > >= == !=`, `and`, `or` and `not` (short-circuiting, as in Python), unary minus, brackets, names, and calls `f(a, b)`.
- **Statements**: `let name = expr;` (create a variable), `name = expr;` (change an existing one), `print expr;`, an expression followed by `;`, a block `{ ... }`, `if (test) statement else statement`, `while (test) statement`, `fn name(params) { ... }` and `return expr;`.
- **Scope**: blocks and functions create new scopes; functions are **closures** that remember the scope they were defined in.
- **Truth**: `nil` and `false` are false; everything else (including `0` and `""`) is true.
- **Built-ins**: `len(string)`, `str(value)` and `type(value)`.
- **Errors** report a line (and a column, when it's known): `line 2, column 11: unexpected character '@'`.

## A plan

An interpreter is a pipeline, just like CPython's (Week 64):

```text
source text --tokenise--> tokens --parse--> syntax tree --evaluate--> output
```

1. The **tokeniser** (today and tomorrow) turns characters into **tokens**: `let x = 3.5;` becomes `LET`, `NAME x`, `OP =`, `NUMBER 3.5`, `OP ;`, `EOF`. It knows nothing about grammar.
2. The **parser** (Wednesday) turns tokens into an **abstract syntax tree** of dataclasses, such as `Let("x", Number(3.5))`, using **recursive descent**: one method per grammar rule, with operator precedence built into which method calls which.
3. The **evaluator** (Thursday) walks the tree, keeping variables in a chain of **environments** (scopes), and produces output.
4. On Friday, **tests** pin it all down and a **profile** shows where it spends its time.

Each stage has its own error class, and every error carries a position so the person writing Tiny code can find the problem. The program grows step by step; each exercise gives you the earlier steps' code, so you can concentrate on the new piece.

## Step 1: the error family

Start, as Week 38 did, with the errors: they shape everything else. Every error the interpreter raises on purpose is a `TinyError`, so a program running Tiny code can catch them all with one `except`, while a genuine bug in the interpreter still shows up as an ordinary Python exception.

## Step 2: showing an error

A position is only useful if the person can see it. Compilers and interpreters print the offending line with a marker under the column, and Python itself has done this for syntax errors for a long time.

## Step 3: tokens and a cursor

A **token** is a small record: what kind of thing it is, the exact text, what it means, and where it started. A **cursor** walks through the source one character at a time and keeps track of the line and column, so every token can record its position.

## Step 4: numbers

Each kind of token gets a small function that starts at the token's first character and reads to its end, leaving the cursor just after it.

## Step 5: strings

Strings are the fiddliest token: they have a beginning and an end that must match, and escapes inside, each of which can be wrong.

## Checkpoint

Here are today's pieces working together on a line of Tiny:

## Quiz

## Before you go

Write a three-line Tiny program on paper and list the tokens you'd expect from it, with their kinds and columns. Tomorrow: the rest of the tokeniser, names, keywords, operators and comments, and the `tokenize` function that ties it together.